In [ ]:
import pandas as pd
from aloud_database.aloud_database import Database

In [ ]:
df_responses = pd.read_csv('data/le24.csv')
db = Database()

In [ ]:
"""
Cell generated by Data Wrangler.
"""
def clean_data(df_responses):
    # Drop column: 'Qual sua faixa etária?'
    df_responses = df_responses.drop(columns=['Qual sua faixa etária?'])
    # Drop column: 'Qual o seu nível de inglês?'
    df_responses = df_responses.drop(columns=['Qual o seu nível de inglês?'])
    # Drop columns: 'utm_source', 'Ending' and 10 other columns
    df_responses = df_responses.drop(columns=['utm_source', 'Ending', 'Tags', 'Network ID', 'Stage Date (UTC)', 'Start Date (UTC)', 'Response Type', 'utm_content', 'utm_term', 'utm_campaign', 'utm_medium', '#'])
    # Rename column 'Qual o seu nome completo?' to 'name'
    df_responses = df_responses.rename(columns={'Qual o seu nome completo?': 'name'})
    # Rename column 'Beleza! Agora, qual o seu melhor e-mail?' to 'email'
    df_responses = df_responses.rename(columns={'Beleza! Agora, qual o seu melhor e-mail?': 'email'})
    # Rename column 'Certo! Qual o seu WhatsApp com DDD?' to 'phone'
    df_responses = df_responses.rename(columns={'Certo! Qual o seu WhatsApp com DDD?': 'phone'})
    # Rename column 'Qual sua renda mensal?' to 'income'
    df_responses = df_responses.rename(columns={'Qual sua renda mensal?': 'income'})
    # Rename column 'O preço do Cronograma dos Fluentes, hoje, é de R$ 1.997,00. Caso surja uma vaga e seu perfil seja aprovado, como você gostaria de prosseguir?\n' to 'caracteristic'
    df_responses = df_responses.rename(columns={'O preço do Cronograma dos Fluentes, hoje, é de R$ 1.997,00. Caso surja uma vaga e seu perfil seja aprovado, como você gostaria de prosseguir?\n': 'caracteristic'})
    # Rename column 'Submit Date (UTC)' to 'submit_date'
    df_responses = df_responses.rename(columns={'Submit Date (UTC)': 'submit_date'})
    # Change column type to datetime64[ns] for column: 'submit_date'
    df_responses = df_responses.astype({'submit_date': 'datetime64[ns]'})
    df_responses['first_name'] = df_responses['name'].apply(lambda x: str(x).split(" ")[0])
    # Capitalize the first character in column: 'name'
    df_responses['name'] = df_responses['name'].str.capitalize()
    # Filter rows based on column: 'income'
    df_responses = df_responses[~df_responses['income'].str.contains("10.000", regex=False, na=False, case=False)]
    df_responses['phone'] = df_responses['phone'].apply(lambda x: str(x).replace("'+",""))

    df_responses = df_responses.drop_duplicates()
    # Drop duplicate rows in column: 'phone'
    df_responses = df_responses.drop_duplicates(subset=['phone'])

    return df_responses

df_responses_clean = clean_data(df_responses.copy())

In [ ]:
def has_purchase_conversion(lead_ids):
    """
    Verifica se algum dos lead_ids possui conversão de compra (conversion_type_id = 8).
    Retorna True se houver pelo menos uma conversão, False caso contrário.
    """
    for lead_id in lead_ids:
        purchase_query = f"""
        SELECT 
            EXISTS (
                SELECT 1
                FROM lead_tracking_prod.conversions c
                WHERE c.lead_id = '{lead_id}'::uuid
                AND c.conversion_type_id = 8
            ) AS has_purchase_conversion;
        """
        result_df = db.execute_query(purchase_query)
        if not result_df.empty and bool(result_df.iloc[0]['has_purchase_conversion']):
            return True
    return False

def locate_lead_by_keys(phone: str = None, email: str = None):
    # Garante que pelo menos um dos parâmetros será passado
    if not phone and not email:
        return {
            "status": "falha",
            "mensagem": "É obrigatório informar ao menos o phone ou o email.",
            "dados": None
        }

    conditions = []
    params = []

    if phone:
        conditions.append(f"(formatted_phone = '{phone}' OR whatsapp_format = '{phone}')")
    if email:
        conditions.append(f"email = '{email}'")

    query = f"""
    SELECT *
        FROM lead_tracking_prod.vw_lead_complete_data
        WHERE {' OR '.join(conditions)};
    """
    df = db.execute_query(query, tuple(params))

    if df.empty:
        return {
            "status": "falha",
            "mensagem": "Lead não localizado.",
            "dados": None
        }

    # Agrupa por lead_id e soma as conversões (convertendo para int)
    df['conversion_count'] = df['conversion_count'].astype(int)
    lead_group = df.groupby('lead_id')['conversion_count'].sum().reset_index()

    # Seleciona o lead_id com maior número de conversões
    lead_id_mais_conversoes = lead_group.sort_values('conversion_count', ascending=False).iloc[0]['lead_id']

    # Filtra o dataframe para o lead_id selecionado
    df_lead = df[df['lead_id'] == lead_id_mais_conversoes]

    # Pega o nome (assumindo que é o mesmo para o lead_id)
    nome = df_lead.iloc[0]['name']

    # Pega o último email cadastrado (maior data de email_created_at)
    df_lead['email_created_at'] = pd.to_datetime(df_lead['email_created_at'])
    ultimo_email = df_lead.sort_values('email_created_at', ascending=False).iloc[0]['email']

    # Verifica se algum dos lead_ids possui conversão de compra
    lead_ids = df['lead_id'].unique()
    possui_compra = has_purchase_conversion(lead_ids)

    return {
        "status": "sucesso",
        "mensagem": "Lead localizado com sucesso.",
        "dados": {
            "lead_id": lead_id_mais_conversoes,
            "name": nome,
            "email": ultimo_email,
            "possui_compra": possui_compra
        }
    }

In [ ]:
sample = df_responses_clean.copy()

# Garante que as colunas existem no DataFrame
for col in ['status', 'lead_id', 'has_purchase', 'name', 'email']:
    if col not in sample.columns:
        sample[col] = None

total = len(sample)
processados = 0

for idx, row in sample.iterrows():
    # Pula se já existe valor não nulo em 'status'
    if pd.notna(row.get('status')):
        continue

    phone = row.get('phone')
    email = row.get('email')
    response = locate_lead_by_keys(phone=phone, email=email)

    # Salva as chaves no DataFrame
    status = response.get('mensagem')
    dados = response.get('dados', {})

    lead_id = dados.get('lead_id') if dados else None
    name = dados.get('name') if dados else None
    email_resp = dados.get('email') if dados else None
    has_purchase = dados.get('possui_compra') if dados else None

    sample.at[idx, 'status'] = status
    sample.at[idx, 'lead_id'] = lead_id
    sample.at[idx, 'has_purchase'] = has_purchase

    # Se não possui name e lead_id foi encontrado, atualiza o name
    
    if pd.isna(row.get('name')) and lead_id:
        sample.at[idx, 'name'] = name

    # Se não possui email e lead_id foi encontrado, atualiza o email
    if pd.isna(row.get('email')) and lead_id:
        sample.at[idx, 'email'] = email_resp

    processados += 1
    print(f"Processados: {processados} de {total}", end='\r')